# 02 — Baseline Model

The simplest model worth taking seriously: LightGBM on the static application table only —
53 numeric A/P columns, no feature engineering. Whatever comes later gets measured against
these numbers.

Scoring uses the competition metric: mean weekly Gini, with a penalty if performance falls
over time.


In [3]:
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from pathlib import Path

DATA = Path("../data/parquet_files")
TRAIN = DATA / "train"

# Load base table
base = pl.read_parquet(TRAIN / "train_base.parquet")

# Load static tables
static = pl.concat([
    pl.read_parquet(TRAIN / "train_static_0_0.parquet"),
    pl.read_parquet(TRAIN / "train_static_0_1.parquet"),
], how="vertical_relaxed")

print(f"Base:   {base.shape}")
print(f"Static: {static.shape}")

Base:   (1526659, 5)
Static: (1526659, 168)


In [4]:
# Select only A and P columns from static
feature_cols = [c for c in static.columns if c.endswith(("A", "P"))]
print(f"Feature columns: {len(feature_cols)}")

# Join base to static
df = base.join(
    static.select(["case_id"] + feature_cols),
    on="case_id",
    how="left"
)

print(f"Joined shape: {df.shape}")
print(f"Default rate: {df['target'].mean():.2%}")

Feature columns: 53
Joined shape: (1526659, 58)
Default rate: 3.14%


## Temporal train/validation split

A random split would leak — in production the model scores future applicants. So the first
80% of weeks go to training and the last 20% to validation.


In [5]:
# Temporal split - 80% train, 20% val by week
weeks = sorted(df["WEEK_NUM"].unique().to_list())
n = len(weeks)

train_weeks = weeks[:int(n * 0.8)]
val_weeks = weeks[int(n * 0.8):]

train_df = df.filter(pl.col("WEEK_NUM").is_in(train_weeks))
val_df = df.filter(pl.col("WEEK_NUM").is_in(val_weeks))

print(f"Total weeks: {n}")
print(f"Train weeks: {min(train_weeks)} to {max(train_weeks)} ({len(train_weeks)} weeks)")
print(f"Val weeks:   {min(val_weeks)} to {max(val_weeks)} ({len(val_weeks)} weeks)")
print(f"\nTrain rows: {len(train_df):,}")
print(f"Val rows:   {len(val_df):,}")
print(f"\nTrain default rate: {train_df['target'].mean():.2%}")
print(f"Val default rate:   {val_df['target'].mean():.2%}")

Total weeks: 92
Train weeks: 0 to 72 (73 weeks)
Val weeks:   73 to 91 (19 weeks)

Train rows: 1,323,314
Val rows:   203,345

Train default rate: 3.29%
Val default rate:   2.19%


In [6]:
X_train = train_df.select(feature_cols).to_pandas()
y_train = train_df["target"].to_pandas()

X_val = val_df.select(feature_cols).to_pandas()
y_val = val_df["target"].to_pandas()

# Keep WEEK_NUM for stability calculation later
val_base = val_df.select(["WEEK_NUM", "target"]).to_pandas()

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"y_train positives: {y_train.sum():,} ({y_train.mean():.2%})")
print(f"y_val positives:   {y_val.sum():,} ({y_val.mean():.2%})")

X_train: (1323314, 53)
X_val:   (203345, 53)
y_train positives: 43,545 (3.29%)
y_val positives:   4,449 (2.19%)


## Training

In [7]:
params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "n_estimators": 1000,
    "is_unbalance": True,
    "verbosity": -1,
    "random_state": 42,
}

lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_val],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100),
    ]
)

Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.778097
[200]	valid_0's auc: 0.782081
[300]	valid_0's auc: 0.782543
Early stopping, best iteration is:
[348]	valid_0's auc: 0.782876


In [8]:
# Predictions
val_preds = model.predict(X_val, num_iteration=model.best_iteration)

# AUC and Gini
auc = roc_auc_score(y_val, val_preds)
gini = 2 * auc - 1
print(f"Val AUC:  {auc:.4f}")
print(f"Val Gini: {gini:.4f}")

Val AUC:  0.7829
Val Gini: 0.5658


## Stability-adjusted Gini (competition metric)

In [9]:
def gini_stability(base_df, preds, w_fallingrate=88.0, w_resstd=-0.5):
    base = base_df.copy()
    base["score"] = preds
    
    gini_in_time = (
        base[["WEEK_NUM", "target", "score"]]
        .sort_values("WEEK_NUM")
        .groupby("WEEK_NUM")[["target", "score"]]
        .apply(lambda x: 2 * roc_auc_score(x["target"], x["score"]) - 1)
        .tolist()
    )
    
    x = np.arange(len(gini_in_time))
    y = np.array(gini_in_time)
    a, b = np.polyfit(x, y, 1)
    residuals = y - (a * x + b)
    
    return {
        "stability_score": float(np.mean(y) + w_fallingrate * min(0, a) + w_resstd * np.std(residuals)),
        "mean_gini": float(np.mean(y)),
        "slope": float(a),
        "residual_std": float(np.std(residuals)),
    }

stability = gini_stability(val_base, val_preds)

print(f"\nStability score: {stability['stability_score']:.4f}")
print(f"Mean gini:       {stability['mean_gini']:.4f}")
print(f"Slope:           {stability['slope']:.6f}")
print(f"Residual std:    {stability['residual_std']:.4f}")


Stability score: 0.5367
Mean gini:       0.5572
Slope:           0.004160
Residual std:    0.0411


## Baseline results

`static_0` only, A+P columns (53 features), no engineering.

| Metric | Value |
|---|---|
| Val AUC | 0.7829 |
| Val Gini | 0.5658 |
| Stability score | 0.5367 |
| Weekly Gini slope | +0.0042 (no falling-rate penalty) |
| Residual std | 0.0411 |

A decent floor. Next: ratio features and depth-1 aggregations, to see how much lift is
actually there.


## Feature importance

In [11]:
importance = pd.DataFrame({
    "feature": model.feature_name(),
    "importance": model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False).head(20)

print(importance.to_string())

                         feature    importance
7    avgdpdtolclosure24_3658938P  1.484267e+06
51             totalsettled_863A  8.319226e+05
22     lastrejectcredamount_222A  6.251796e+05
47                   price_1097A  5.647503e+05
14                  currdebt_22A  2.147686e+05
37          maxdpdtolerance_374P  2.090991e+05
13               credamount_770A  2.067775e+05
24               maxannuity_159A  2.061778e+05
16     disbursedcredamount_1113A  1.978145e+05
4      avgdbddpdlast24m_3658932P  1.808091e+05
2                   annuity_780A  1.635948e+05
49  sumoutstandtotalest_4493215A  1.395954e+05
32            maxdpdlast12m_727P  1.304370e+05
48     sumoutstandtotal_3546847A  1.122184e+05
33            maxdpdlast24m_143P  1.108921e+05
27  maxdbddpdtollast12m_3658940P  1.021079e+05
34             maxdpdlast3m_392P  9.469132e+04
18    inittransactionamount_650A  9.287921e+04
30    maxdpdfrom6mto36m_3546853P  7.933927e+04
6      avgdbdtollast24m_4525197P  7.444544e+04
